# Fiabilisation de la base Diplômes — v13

Même démarche que la v12, en plus léger : chaque étape = **règle** (une table ou une fonction, modifiable) → **code** → **un contrôle**.

| Étape | Contenu | Où modifier |
|---|---|---|
| 0 | Chargement | `FICHIER_SOURCE`, `FICHIER_SORTIE`, `FICHIER_REF_ECOLES` |
| 1 | Nettoyage technique | `PLACEHOLDERS` |
| 2 | Niveau saisi → niveau standard | `NIVEAUX`, `NIVEAUX_ECRITS` |
| 3 | Libellé de diplôme → type de diplôme, niveau théorique, indice de catégorie | `REF_DIPLOMES_MANUEL`, `REGLES_DIPLOME`, `FOURCHETTES_NIVEAU`, `INDICES_CATEGORIE` |
| 4 | Niveau fiabilisé | `decider_niveau`, `ARRONDI_NIVEAU` |
| 5 | École (référentiel Excel) et catégorie | `ref_ecoles.xlsx`, `CATEGORIE_PAR_DEFAUT`, `CATEGORIE_INCERTAINE_SANS_ECOLE` |
| 6 | Dédoublonnage, doublons partiels, quasi-doublons | `CLES_DOUBLON` |
| 7 | Contrôles et statut | `CONTROLES`, `PRIORITE_STATUT` |
| 8 | Vue salarié : valeurs sûres, sinon neutre + motif | `NIVEAU_SI_INCERTAIN`, `CATEGORIE_SI_AMBIGU` |
| 9 | Export | `onglets` |

**Ce qui change par rapport à la v12**

- Le référentiel des écoles (330 lignes de code) est dans **`ref_ecoles.xlsx`**, à côté du notebook : il se relit et se complète dans Excel. Les codes inconnus sortent dans l'onglet `Ecoles_a_completer` du fichier de sortie.
- Décisions métier intégrées : brevet bancaire et certificat d'aptitude bancaire = certificats (niveau `AUTRES`, le niveau saisi est ignoré) ; ITB = BAC+4 ; MBA et Executive MBA / Master = `AUTRES` (hors du calcul du diplôme le plus élevé) ; **Diplôme des métiers de banque = `AUTRES`** avec le flag `NIVEAU_SANS_EQUIVALENCE` ; un bac saisi « niveau secondaire » ou « niveau bac » n'est plus signalé (convention du SIRH).
- **Dédoublonnage** : la ligne gardée est la mieux renseignée (plus « la première ») ; une ligne qui ne fait que répéter une autre avec moins d'informations (même type de diplôme, même niveau, spécialité ou école vides) est retirée comme **doublon partiel** ; les **quasi-doublons** sont cherchés sur le type de diplôme (`ITB` = `INSTITUT DES TECHNIQUES DE BANQUE`, `BAC` = `BACCALAUREAT`, `BTS` = `TECHNICIEN SPECIALISE`…), plus sur le libellé exact.
- Nouveau flag `LIBELLE_NON_DIPLOMANT` : le texte n'est pas un diplôme (exigence de poste « Bac+2/3 avec 5 ans d'exp ou Bac+4/5 », type d'école « Grande école de commerce », domaine seul « Finance »).
- **Un principe pour tous les choix** : chaque ligne dit si son niveau est `SUR` ou `INCERTAIN` (`niveau_fiabilite`, `motif_niveau`). Seules les lignes sûres comptent pour le salarié. **`Vue_salarie` ne contient que les salariés dont tout est sûr** ; les autres vont dans **`Salaries_a_discuter`** avec `A VERIFIER` (niveau) ou `AUTRES` (catégorie) et le motif (`motif_niveau_max`, `motif_categorie_max`). Ex. deux « DIPLOME » à l'université + un bac : à discuter, « au moins le bac », on ne devine pas.
- `type_diplome` est exporté (il sert aux quasi-doublons et à la relecture).

**Relancer sur une nouvelle extraction** : déposer l'Excel à côté du notebook, changer `FICHIER_SOURCE`, *Run > Run All Cells*. Trois listes à regarder ensuite : niveaux écrits inconnus (étape 2), libellés non classés (étape 3), écoles hors référentiel (étape 5) — ce sont les seules nouveautés qui demandent une décision.

In [1]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

# Seules lignes à changer pour une nouvelle extraction (fichiers lus et écrits dans le dossier du notebook)
FICHIER_SOURCE = "Diplômes à fiabiliser - CB 02112026.xlsx"
FICHIER_SORTIE = "BDD_Diplomes_FINAL.xlsx"
FICHIER_REF_ECOLES = "ref_ecoles.xlsx"

---
## Étape 0 : Chargement

Colonnes renommées en noms courts ; `ligne_source` = numéro de ligne Excel d'origine, gardé jusqu'au bout pour revenir au fichier initial. Si le fichier n'est pas trouvé, la cellule liste les Excel du dossier (le nom compare sans tenir compte de la casse ni de la façon dont les accents sont encodés).

In [2]:
COLONNES = {
    "Matricule fictif": "matricule",
    "Nom de diplôme ou certification": "diplome",
    "Niveau du diplôme": "niveau",
    "Spécialité du diplôme": "specialite",
    "Ecole Diplome ou Organisme Certification": "ecole",
}
COLS = list(COLONNES.values())

def trouver_fichier(nom, dossiers=(Path.cwd(), Path.cwd() / "fiabilisation_diplomes")):
    # Un même « ô » peut être encodé de deux façons (fichier passé par un Mac) : on compare en NFC et sans casse
    def meme_nom(a, b):
        return unicodedata.normalize("NFC", a).casefold() == unicodedata.normalize("NFC", b).casefold()
    dossiers = [d for d in dossiers if d.is_dir()]
    chemin = next((f for d in dossiers for f in d.iterdir() if f.is_file() and meme_nom(f.name, nom)), None)
    if chemin is None:
        excels = sorted({f.name for d in dossiers for f in d.glob("*.xls*") if not f.name.startswith("~$")})
        raise FileNotFoundError(f"« {nom} » introuvable dans {Path.cwd()}. Fichiers Excel présents : {', '.join(excels) or 'aucun'}.")
    return chemin

CHEMIN_SOURCE = trouver_fichier(FICHIER_SOURCE)
DOSSIER = CHEMIN_SOURCE.parent
brut = pd.read_excel(CHEMIN_SOURCE)
manquantes = [c for c in COLONNES if c not in brut.columns]
assert not manquantes, f"Colonnes absentes du fichier : {manquantes}"

df = brut.rename(columns=COLONNES)[COLS].copy()
df["ligne_source"] = df.index + 2
print(f"{CHEMIN_SOURCE.name} : {len(df):,} lignes, {df['matricule'].nunique():,} matricules, "
      f"{df.duplicated(subset=COLS).sum():,} doublons stricts".replace(",", " "))
pd.DataFrame({"valeurs_distinctes": df[COLS].nunique(), "lignes_vides": df[COLS].isna().sum(),
              "% vides": (df[COLS].isna().mean() * 100).round(1)})

Diplômes à fiabiliser - CB 02112026.xlsx : 27 553 lignes  8 864 matricules  6 401 doublons stricts


,valeurs_distinctes,lignes_vides,% vides
matricule,8864,0,0.0
diplome,214,0,0.0
niveau,25,11799,42.8
specialite,83,1031,3.7
ecole,334,1149,4.2


---
## Étape 1 : Nettoyage technique

Pour que la forme ne trompe pas les étapes suivantes : MAJUSCULES, sans accents, espaces simples, espaces normalisés autour de `/ & , :`, niveau sans espace autour du `+` (`Bac + 3` → `BAC+3`, `Bac + 3 ou 4` → `BAC+3 OU BAC+4`). Les **vides déguisés** (`-`, `AUTRES`, `DEFAUT`, `N/A`…) deviennent de vrais vides : « Autres » ne dit rien sur l'école ou la spécialité, et un vide se voit dans les contrôles. Le matricule n'est pas touché.

In [3]:
PLACEHOLDERS = {"", "-", "--", ".", "?", "NA", "N/A", "NAN", "NONE", "NULL",
                "AUTRE", "AUTRES", "DEFAUT", "INCONNU", "NON RENSEIGNE", "A COMPLETER"}

def normaliser_texte(x):
    if pd.isna(x):
        return None
    x = re.sub(r"[‘’ʼ´`]", "'", str(x))
    s = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode()
    s = re.sub(r"\s*([/&,:])\s*", r" \1 ", s)
    s = re.sub(r"\s+", " ", s).strip().upper()
    return None if s in PLACEHOLDERS else s

for c in ["diplome", "niveau", "specialite", "ecole"]:
    df[f"{c}_norm"] = df[c].map(normaliser_texte)
df["niveau_norm"] = (df["niveau_norm"].str.replace(r"\s*\+\s*", "+", regex=True)
                                      .str.replace(r"^BAC\+(\d) OU (\d)$", r"BAC+\1 OU BAC+\2", regex=True))

pd.DataFrame({c: {"distinct_avant": df[c].nunique(), "distinct_apres": df[f"{c}_norm"].nunique(),
                  "vides_avant": df[c].isna().sum(), "vides_apres": df[f"{c}_norm"].isna().sum()}
              for c in ["diplome", "niveau", "specialite", "ecole"]}).T

,distinct_avant,distinct_apres,vides_avant,vides_apres
diplome,214,201,0,1
niveau,25,18,11799,11809
specialite,83,80,1031,2475
ecole,334,319,1149,2861


---
## Étape 2 : Niveau saisi → niveau standard

Rangs : 0 SECONDAIRE · 1 BAC · 2 BAC+1 · 3 BAC+2 · 4 BAC+3 · 5 BAC+4 · 6 BAC+5 et BAC+6 · 7 BAC+7 (doctorat) et expertise comptable. `AUTRES` n'a pas de rang : certificats, MBA et Executive MBA / Master, diplômes sans équivalence établie ; il vient toujours du libellé (étape 3), jamais de la saisie, et ne compte pas pour le diplôme le plus élevé.

Pour chaque écriture : **devient** (le niveau standard), **accepte de … à** (les niveaux de diplôme compatibles, étape 4), **imprécis** (signalé à titre informatif), **signaler** (`False` = convention du SIRH, la différence avec le libellé n'est pas signalée : un bac saisi « niveau secondaire » ou « niveau bac », un master saisi « Bac + 6 »).

| Écriture | Devient | Accepte | Note |
|---|---|---|---|
| `NIVEAU SECONDAIRE`, `CAP` | SECONDAIRE | SECONDAIRE → BAC | convention : un bac saisi ainsi n'est pas signalé |
| `NIVEAU BAC` | SECONDAIRE | SECONDAIRE → BAC | allé jusqu'au bac sans dire qu'il l'a eu |
| `NIVEAU UNIVERSITAIRE` | BAC | BAC → BAC+7 | imprécis : on sait seulement qu'il a le bac |
| `NIVEAU SUPERIEUR` | BAC+3 | BAC+3 → BAC+7 | imprécis |
| `BAC+n ET PLUS`, `>BAC+5` | BAC+n ET PLUS | BAC+n → BAC+7 | imprécis ; arrondi à BAC+n à l'étape 4 |
| `BAC+3 OU BAC+4` | BAC+3 OU BAC+4 | BAC+3 → BAC+4 | imprécis ; arrondi à BAC+3 |
| `BAC+6` | BAC+6 | BAC+5 → BAC+7 | saisi surtout sur des masters, même rang que BAC+5 |

Une écriture absente de la table est listée à l'exécution : l'ajouter dans `NIVEAUX_ECRITS`.

In [4]:
NIVEAUX = {
    "SECONDAIRE": 0, "BAC": 1, "BAC+1": 2,
    "BAC+2": 3, "BAC+2 ET PLUS": 3,
    "BAC+3": 4, "BAC+3 OU BAC+4": 4,
    "BAC+4": 5, "BAC+4 ET PLUS": 5,
    "BAC+5": 6, "BAC+5 ET PLUS": 6, "BAC+6": 6, "BAC+6 ET PLUS": 6,
    "BAC+7": 7, "EXPERTISE COMPTABLE": 7,
}
RANG_VERS_NIVEAU = {}
for niv, rang in NIVEAUX.items():
    RANG_VERS_NIVEAU.setdefault(rang, niv)          # rang -> niveau simple (le premier listé)

#                           devient            accepte de     à            imprécis  signaler
NIVEAUX_ECRITS = {
    "SECONDAIRE":           ("SECONDAIRE",     "SECONDAIRE", "SECONDAIRE", False, True),
    "NIVEAU SECONDAIRE":    ("SECONDAIRE",     "SECONDAIRE", "BAC",        False, False),
    "CAP":                  ("SECONDAIRE",     "SECONDAIRE", "BAC",        False, False),
    "BAC":                  ("BAC",            "BAC",        "BAC",        False, True),
    "NIVEAU BAC":           ("SECONDAIRE",     "SECONDAIRE", "BAC",        False, False),
    "NIVEAU UNIVERSITAIRE": ("BAC",            "BAC",        "BAC+7",      True,  True),
    "NIVEAU SUPERIEUR":     ("BAC+3",          "BAC+3",      "BAC+7",      True,  True),
    "BAC+1":                ("BAC+1",          "BAC+1",      "BAC+1",      False, True),
    "BAC+2":                ("BAC+2",          "BAC+2",      "BAC+2",      False, True),
    "BAC+2 ET PLUS":        ("BAC+2 ET PLUS",  "BAC+2",      "BAC+7",      True,  True),
    "BAC+3":                ("BAC+3",          "BAC+3",      "BAC+3",      False, True),
    "BAC+3 OU BAC+4":       ("BAC+3 OU BAC+4", "BAC+3",      "BAC+4",      True,  True),
    "BAC+4":                ("BAC+4",          "BAC+4",      "BAC+4",      False, True),
    "BAC+4 ET PLUS":        ("BAC+4 ET PLUS",  "BAC+4",      "BAC+7",      True,  True),
    "BAC+5":                ("BAC+5",          "BAC+5",      "BAC+5",      False, True),
    "BAC+5 ET PLUS":        ("BAC+5 ET PLUS",  "BAC+5",      "BAC+7",      True,  True),
    ">BAC+5":               ("BAC+5 ET PLUS",  "BAC+5",      "BAC+7",      True,  True),
    "BAC+6":                ("BAC+6",          "BAC+5",      "BAC+7",      False, False),
}
assert all(n in NIVEAUX for v in NIVEAUX_ECRITS.values() for n in v[:3]), "un niveau de NIVEAUX_ECRITS manque dans NIVEAUX"

inconnus = sorted(set(df["niveau_norm"].dropna()) - set(NIVEAUX_ECRITS))
print("Niveaux écrits inconnus (à ajouter dans NIVEAUX_ECRITS) :", inconnus or "aucun")

df["niveau_normalise"] = df["niveau_norm"].map(lambda v: NIVEAUX_ECRITS[v][0] if v in NIVEAUX_ECRITS else None)
df["ctrl_niveau_saisi_imprecis"] = df["niveau_norm"].map(lambda v: NIVEAUX_ECRITS[v][3] if v in NIVEAUX_ECRITS else pd.notna(v)).astype(bool)

(df.groupby(["niveau", "niveau_normalise"], dropna=False).size().rename("nb_lignes").reset_index()
   .assign(rang=lambda d: d["niveau_normalise"].map(NIVEAUX)).sort_values(["rang", "nb_lignes"], ascending=[True, False])
   .drop(columns="rang").reset_index(drop=True))

Niveaux écrits inconnus (à ajouter dans NIVEAUX_ECRITS) : aucun


,niveau,niveau_normalise,nb_lignes
0,NIVEAU SECONDAIRE,SECONDAIRE,2225
1,SECONDAIRE,SECONDAIRE,29
2,NIVEAU BAC,SECONDAIRE,8
3,CAP,SECONDAIRE,1
4,BAC,BAC,1355
5,Bac,BAC,906
6,NIVEAU UNIVERSITAIRE,BAC,37
7,Bac + 1,BAC+1,7
8,Bac +1,BAC+1,6
9,BAC+2,BAC+2,2399


---
## Étape 3 : Libellé de diplôme → type de diplôme, niveau théorique, indice de catégorie

Des règles par mots-clés testées dans l'ordre, du plus précis au plus général ; `REF_DIPLOMES_MANUEL` passe avant les règles. Chaque libellé reçoit un **type de diplôme** (`type_diplome` : MASTER, LICENCE, ITB…), un **niveau théorique** et, plus bas, un **indice de catégorie**.

| Niveau | Types de diplôme |
|---|---|
| BAC+7 | Doctorat ; expertise comptable (niveau à part, même rang) |
| BAC+5 | Master, Master 2, Ingénieur, Grande école, DESS / DEA / DESA, Cycle supérieur |
| BAC+4 | Master 1, Maîtrise, Licence avant 2006, Cycle normal, **ITB**, Bachelor 4 ans, Ingénieur d'application |
| BAC+3 | Licence, Bachelor 3 ans, Infirmier |
| BAC+2 | DEUG, DUT ; BTS, Technicien, Technicien spécialisé ; Classes prépa |
| BAC / SECONDAIRE | Baccalauréat / Secondaire, Niveau bac, CAP, formation qualifiante |
| AUTRES | **Certificats** (dont brevet bancaire, certificat d'aptitude bancaire) ; **MBA, Executive MBA / Master** ; **Diplôme des métiers de banque** (sans équivalence : flag `NIVEAU_SANS_EQUIVALENCE`) |
| fourchette | Technicien spécialisé / Licence pro (BAC+2–3) ; Licence / Master (BAC+3–5) ; Bac+2/3 … Bac+4/5 (BAC+2–5) ; 3ème cycle (BAC+5–7) ; DES (BAC+4–5, BAC+5 sans saisie) |
| sans niveau | libellés vagues (`DIPLOME`, `UNIVERSITE`), domaine seul (`Marketing`), degrés comptables, CET, brevet professionnel : le niveau saisi est gardé |

**Libellés non diplômants** (`LIBELLE_NON_DIPLOMANT`, informatif) : exigences de poste (« … ou équivalent », « avec 5 ans d'expérience »), types d'école (« Grande école de commerce »), domaine seul, libellé générique. Le niveau est calculé normalement, mais ces lignes ne sont jamais choisies comme diplôme le plus élevé si le salarié a une autre ligne au même niveau.

In [5]:
TYPE_PAR_DEFAUT = "DIPLOME NON IDENTIFIE"
GENERIQUE, DOMAINE = "DIPLOME GENERIQUE (A PRECISER)", "DOMAINE SEUL (NIVEAU NON PRECISE)"

# Décisions libellé par libellé (clé = libellé nettoyé) ; None = pas de niveau justifiable, on garde la saisie
REF_DIPLOMES_MANUEL = {
    "INFIRMERIE": ("LICENCE", "BAC+3"),
    "INSTITUT SUPERIEUR D'ENTREPRISE": ("LICENCE", "BAC+3"),
    "GRANDES ECOLES DE COMMERCE / MANAGEMENT": ("GRANDE ECOLE (COMMERCE/INGENIEUR)", "BAC+5"),
    "INGENIERIE FISCALE COMPTABLE ET FINANCIERE": ("MASTER", "BAC+5"),
    "ECOLE SUP.DES INDUSTRIES TEXTILE & HABIL": ("INGENIEUR", "BAC+5"),            # ESITH (décision métier)
    "CET SECRETARIAT": ("CERTIFICAT D'ENSEIGNEMENT TECHNIQUE (CET)", None),
    "CERTIFICAT D'ETUDES SUPERIEURES": ("CERTIFICAT D'ETUDES SUPERIEURES", None),
    "DIPLOME DES ETUDES COMPTABLES SPECIALISE": ("DIPLOME COMPTABLE (DEGRE)", None),
    "ACADEMIQUE": (GENERIQUE, None), "DIPLOME": (GENERIQUE, None), "DIPLOME UNIVERSITAIRE": (GENERIQUE, None),
    "DIPLOME UNIVERSITE": (GENERIQUE, None), "UNIVERSITE": (GENERIQUE, None),
}

# (regex, type de diplôme, niveau théorique) : la première règle qui trouve son motif décide
REGLES_DIPLOME = [
    # Certificats : niveau AUTRES ; brevet bancaire et certificat d'aptitude bancaire en font partie (décision métier)
    (r"^CERTIFICAT$|^ATTEST|CERTIFICATION|SCRUM MASTER|BREVET BANCAIRE|APTITUDE BANCAIRE", "CERTIFICATION", "AUTRES"),
    # Diplôme des métiers de banque (CFPB) : pas d'équivalence établie -> AUTRES, signalé NIVEAU_SANS_EQUIVALENCE
    (r"METIERS DE BANQUE", "DIPLOME DES METIERS DE BANQUE (SANS EQUIVALENCE)", "AUTRES"),
    # Libellés à fourchette (voir FOURCHETTES_NIVEAU)
    (r"TECHNICIEN\s*SPECIALISE\s*/\s*LICENCE|DUT\s*,\s*BTS\s*,\s*LICENCE", "TECHNICIEN SPECIALISE OU LICENCE PRO", None),
    (r"LICENCE\s*/\s*MASTER", "LICENCE OU MASTER", None),
    (r"BAC\s*\+?\s*2\s*/\s*3\s+AVEC\s+.*BAC\s*\+?\s*4\s*/\s*5", "BAC+2 A BAC+5 (DIPLOME NON PRECISE)", None),
    (r"(3\s*E(ME)?|TROISIEME) CYCLE", "3EME CYCLE", None),
    # Classes prépa = BAC+2 (décision métier), même si le libellé cite l'école de commerce
    (r"CLASSES PREPARATOIRES", "CLASSES PREPARATOIRES", "BAC+2"),
    # Libellé « BAC+n » ou « NIVEAU BAC+n » : le niveau est écrit
    (r"^(NIVEAU )?BAC\s*\+\s*1", "NIVEAU BAC+1 (DIPLOME NON PRECISE)", "BAC+1"),
    (r"^(NIVEAU )?BAC\s*\+\s*2", "NIVEAU BAC+2 (DIPLOME NON PRECISE)", "BAC+2"),
    (r"^(NIVEAU )?BAC\s*\+\s*3", "NIVEAU BAC+3 (DIPLOME NON PRECISE)", "BAC+3"),
    (r"^(NIVEAU )?BAC\s*\+\s*4", "NIVEAU BAC+4 (DIPLOME NON PRECISE)", "BAC+4"),
    (r"^(NIVEAU )?BAC\s*\+\s*5", "NIVEAU BAC+5 (DIPLOME NON PRECISE)", "BAC+5"),
    (r"^(NIVEAU )?BAC\s*\+\s*[6-9]", "NIVEAU BAC+6 ET PLUS (DIPLOME NON PRECISE)", "BAC+6 ET PLUS"),
    (r"^NIVEAU BAC\s*\+", "NIVEAU D'ETUDES (DIPLOME NON PRECISE)", None),
    # Diplômes, par niveau décroissant
    (r"DOCTORAT|^PH\.?\s?D\b|\bPHD\b|DOCTEUR", "DOCTORAT", "BAC+7"),
    (r"EXECUTI(VE|F) (MASTER|MBA)|(MASTER|MBA) EXECUTI(VE|F)|\bEMBA\b", "MASTER EXECUTIVE", "AUTRES"),
    (r"\bMASTER\s*-?\s*(1|I)(?![\w.])(?!\s*(ET|/|&|,)\s*(2|II)\b)|\bM1\b|1ERE ANNEE (DE )?MASTER|MASTER\s*\(?\s*1ERE ANNEE", "MASTER 1", "BAC+4"),
    (r"\bMASTER\s*-?\s*(2|II)(?![\w.])|\bM2\b|\bMASTER\s*1\s*(ET|/|&|,)\s*2\b", "MASTER", "BAC+5"),
    (r"\bE?MBA\b|M\.B\.A|MASTER OF BUSINESS", "MBA", "AUTRES"),                       # MBA = AUTRES, comme l'Executive MBA (décision métier)
    (r"\bMASTERE?S?\b|^TYPE MASTER", "MASTER", "BAC+5"),
    (r"^(?!.*\b(LICENCE|BTS|DUT|PREPA|TECHNICIEN|BACH?E?LOR)\b)(ESC\b|.*(GRANDES? ECOLES?|ECOLES? (SUPERIEURE )?DE COMMERCE|ECOLE COMMERCE|ECOLES? D.INGE))",
     "GRANDE ECOLE (COMMERCE/INGENIEUR)", "BAC+5"),
    (r"INGENIEUR D.APPLICATION", "INGENIEUR D'APPLICATION", "BAC+4"),
    (r"INGENIEUR|^(DIPLOME D.)?INGENIERIE|^ING\b", "INGENIEUR", "BAC+5"),
    (r"^DIPLOME (DES |D.)ETUDES SUPERIEURE?S\b(?!\s*(SPECIAL|APPLIQU|APPROFONDI))", "DES (DIPLOME DES ETUDES SUPERIEURES)", None),
    (r"EXPERTISE COMPTABLE|EXPERT[ -]COMPTABLE", "EXPERTISE COMPTABLE", "EXPERTISE COMPTABLE"),
    (r"CYCLE SUPERIEUR", "DIPLOME CYCLE SUPERIEUR", "BAC+5"),                         # ISCAE / ENCG
    (r"\bD\.?E\.?S\.?S\.?(?!\w)|\bD\.?E\.?S\.?A\.?(?!\w)|ETUDES SUPERIEURES SPECIAL|ETUDES APPROFONDIES|^DEA\b|ETUDES SUPERIEURES APPLIQUEE",
     "DIPLOME 3E CYCLE (DES/DESS/DEA/DESA)", "BAC+5"),
    (r"MAITRISE", "MAITRISE", "BAC+4"),
    (r"LICENCE ANTERIEURE A 2006", "LICENCE (ANCIEN REGIME 4 ANS)", "BAC+4"),
    (r"CYCLE NORMAL", "DIPLOME CYCLE NORMAL", "BAC+4"),                               # ISCAE / ENCG
    (r"INSTITUT (DES )?TECHNIQUES? (DE )?BANQUE|\bITB\b", "ITB", "BAC+4"),           # décision métier : BAC+4
    (r"BACH?E?LOR\s*BAC\s*\+\s*4", "BACHELOR 4 ANS", "BAC+4"),
    (r"\d\s*EME DEGRES COMPT", "DIPLOME COMPTABLE (DEGRE)", None),
    (r"LICENCE|LICENSE|LICENCIE|EQUIVAL\.?LIC|BACH?E?LOR\s*BAC\s*\+\s*3|INFIRM|\bBACH?E?LOR\b", "LICENCE", "BAC+3"),
    (r"D\.?E\.?U\.?G|ETUDES UNIVERSITAIRES G|\bD\.?U\.?T\b|UNIVERSITAIRE DE TECHNOLOGIE|\bDEUST\b", "DEUG / DUT", "BAC+2"),
    (r"\bB\.?T\.?S\b|BREVET DE TECHNICIEN SUP|BREVET TECHNICIEN|TECHNICIEN", "BTS / TECHNICIEN SPECIALISE", "BAC+2"),
    (r"^NIVEAU BAC(CALAUREAT)?$", "SECONDAIRE / QUALIFIANT", "SECONDAIRE"),
    (r"BACCALAUREAT|^BAC\b(?!\s*\+)", "BACCALAUREAT", "BAC"),
    (r"BREVET PROFESSIONNEL", "BREVET PROFESSIONNEL", None),
    (r"SECONDAIRE|APTITUDE PROFESSIONNELLE|FORMATION PROFESSIONNELLE|FORMATION PRO\b|^CAP\b", "SECONDAIRE / QUALIFIANT", "SECONDAIRE"),
    # Intitulés sans niveau : « Diplôme en… » ou domaine seul -> le niveau saisi est gardé
    (r"^DIPLOME EN ", GENERIQUE, None),
    (r"FINANCE|MARKETING|COMMERCE|GESTION|PSYCHOLOGIE|RESSOURCES HUMAINES|COMPTABILITE|AUDIT|BANQUE", DOMAINE, None),
]

# Types de diplôme à fourchette : (plancher, plafond) ; sans saisie : le plancher, ou DEFAUT_FOURCHETTE
FOURCHETTES_NIVEAU = {
    "TECHNICIEN SPECIALISE OU LICENCE PRO": ("BAC+2", "BAC+3"),
    "LICENCE OU MASTER": ("BAC+3", "BAC+5"),
    "BAC+2 A BAC+5 (DIPLOME NON PRECISE)": ("BAC+2", "BAC+5"),
    "3EME CYCLE": ("BAC+5", "BAC+7"),
    "DES (DIPLOME DES ETUDES SUPERIEURES)": ("BAC+4", "BAC+5"),
}
DEFAUT_FOURCHETTE = {"DES (DIPLOME DES ETUDES SUPERIEURES)": "BAC+5"}   # 70 % des saisies

# Types de diplôme dont le libellé ne désigne pas un diplôme précis
TYPES_NON_CLASSES = {TYPE_PAR_DEFAUT, GENERIQUE, DOMAINE, "NIVEAU D'ETUDES (DIPLOME NON PRECISE)"}
TYPES_NON_DIPLOMANTS = TYPES_NON_CLASSES | {"GRANDE ECOLE (COMMERCE/INGENIEUR)", "BAC+2 A BAC+5 (DIPLOME NON PRECISE)"}
# Texte d'exigence de poste (« … ou équivalent », « avec 5 à 10 ans d'exp », « DEA, MBA, école d'ingénieur ou de commerce »)
EXIGENCE_POSTE = r"EQUIVALENT|AVEC .*\bANS\b|CADRE AVEC|ET / OU|\bOU (?:DE |D')?(?:COMMERCE|INGENIEUR|MBA|MASTER|ECONOMIE|NIVEAU)"

def classer_diplome(lib):
    if lib in REF_DIPLOMES_MANUEL:
        return REF_DIPLOMES_MANUEL[lib]
    for regex, type_dipl, niveau in REGLES_DIPLOME:
        if re.search(regex, lib):
            return (type_dipl, niveau)
    return (TYPE_PAR_DEFAUT, None)

classement = {lib: classer_diplome(lib) for lib in df["diplome_norm"].dropna().unique()}
df["type_diplome"] = df["diplome_norm"].map(lambda l: classement[l][0] if pd.notna(l) else None)
df["niveau_theorique"] = df["diplome_norm"].map(lambda l: classement[l][1] if pd.notna(l) else None)
df["type_ligne"] = np.where(df["type_diplome"].eq("CERTIFICATION"), "CERTIFICATION", "DIPLOME")

df["ctrl_diplome_manquant"] = df["diplome_norm"].isna()
df["ctrl_libelle_ambigu"] = df["type_diplome"].isin(FOURCHETTES_NIVEAU)
df["ctrl_diplome_non_classe"] = df["type_diplome"].isin(TYPES_NON_CLASSES)
df["ctrl_libelle_non_diplomant"] = (df["type_diplome"].isin(TYPES_NON_DIPLOMANTS)
                                    | df["diplome_norm"].fillna("").str.contains(EXIGENCE_POSTE, regex=True))
df["ctrl_niveau_sans_equivalence"] = df["type_diplome"].str.contains("SANS EQUIVALENCE", na=False)

print(f"{len(classement)} libellés distincts -> {df['type_diplome'].nunique()} types de diplôme")
(df.groupby(["type_diplome", "niveau_theorique"], dropna=False)
   .agg(libelles=("diplome_norm", "nunique"), nb_lignes=("ligne_source", "size")).reset_index()
   .assign(rang=lambda d: d["niveau_theorique"].map(NIVEAUX)).sort_values(["rang", "nb_lignes"], ascending=[False, False])
   .drop(columns="rang").reset_index(drop=True))

201 libellés distincts -> 38 types de diplôme


,type_diplome,niveau_theorique,libelles,nb_lignes
0,DOCTORAT,BAC+7,2,25
1,EXPERTISE COMPTABLE,EXPERTISE COMPTABLE,1,7
2,MASTER,BAC+5,55,4282
3,INGENIEUR,BAC+5,8,752
4,DIPLOME 3E CYCLE (DES/DESS/DEA/DESA),BAC+5,6,258
5,DIPLOME CYCLE SUPERIEUR,BAC+5,1,92
6,GRANDE ECOLE (COMMERCE/INGENIEUR),BAC+5,14,65
7,DIPLOME CYCLE NORMAL,BAC+4,1,692
8,LICENCE (ANCIEN REGIME 4 ANS),BAC+4,1,547
9,MAITRISE,BAC+4,2,286


In [6]:
# Contrôle : libellés non classés (à relire : REF_DIPLOMES_MANUEL ou REGLES_DIPLOME)
(df[df["ctrl_diplome_non_classe"]].groupby(["diplome", "type_diplome"]).size()
   .sort_values(ascending=False).rename("nb_lignes").reset_index())

,diplome,type_diplome,nb_lignes
0,DIPLOME UNIVERSITAIRE,DIPLOME GENERIQUE (A PRECISER),117
1,DIPLOME,DIPLOME GENERIQUE (A PRECISER),94
2,Diplome Université,DIPLOME GENERIQUE (A PRECISER),10
3,Commerce international,DOMAINE SEUL (NIVEAU NON PRECISE),5
4,Diplôme en Audit/Contrôle de Gestion,DIPLOME GENERIQUE (A PRECISER),5
5,Université,DIPLOME GENERIQUE (A PRECISER),5
6,marketing / communication / digital,DOMAINE SEUL (NIVEAU NON PRECISE),5
7,Commerce / finance,DOMAINE SEUL (NIVEAU NON PRECISE),4
8,Finance/Audit/Contrôle et gestion des risques,DOMAINE SEUL (NIVEAU NON PRECISE),4
9,"Finance, Gestion, Management…",DOMAINE SEUL (NIVEAU NON PRECISE),3


**Indice de catégorie** (`categorie_libelle`) : de quoi parle le libellé — ingénieur, commerce, université, ou autre chose (bac, BTS, ITB, certificat). S'il n'en cite qu'une, c'est son indice ; plusieurs (« Diplôme d'ingénieur / Master école de commerce ») ou aucune (« Diplôme », « Cycle normal ») : pas d'indice, l'école décidera (étape 5). Un libellé de niveau `AUTRES` a toujours l'indice `AUTRES`. « Commerce » seul est un domaine (« commerce international »), pas une école de commerce.

In [7]:
CAT_ING, CAT_COM, CAT_UNIV, CAT_AUTRES = "DIPLOME D'INGENIEUR", "DIPLOME DE COMMERCE", "DIPLOME UNIVERSITAIRE", "AUTRES"

INDICES_CATEGORIE = {
    CAT_ING: r"INGENIEUR|ECOLES? D.INGE|^ING\b|^(DIPLOME D.)?INGENIERIE\b(?!\s+(FISCALE|FINANCIERE))",
    CAT_COM: r"ECOLES? (SUPERIEURES? )?(DE )?COMMERCE|(/|,|\bOU) (DE )?COMMERC|\bE?MBA\b|M\.B\.A|MASTER OF BUSINESS|BUSINESS SCHOOL|^ESC\b",
    CAT_UNIV: (r"UNIVERSIT|\bMASTERE?S?\b(?! OF BUSINESS)|\bM[12]\b|LICENCE|LICENSE|LICENCIE|MAITRISE|DOCTORAT|\bPH\.?\s?D\b|DOCTEUR"
               r"|D\.?E\.?U\.?G|\bD\.?U\.?T\b|\bDEUST\b|\bD\.?E\.?S\.?S\b|\bD\.?E\.?A\b|\bD\.?E\.?S\.?A\b|ETUDES SUPERIEURE?S|ETUDES APPROFONDIES"
               r"|(3\s*E(ME)?|TROISIEME) CYCLE|BACH?E?LOR|INFIRM"),
    CAT_AUTRES: (r"BACCALAUREAT|^BAC\b(?!\s*\+)|^NIVEAU BAC$|SECONDAIRE|TECHNICIEN|\bB\.?T\.?S\b|BREVET|APTITUDE|METIERS DE BANQUE"
                 r"|TECHNIQUES? (DE )?BANQUE|\bITB\b|CERTIFICAT|ATTEST|FORMATION PRO|QUALIFIANT|^CAP\b|^CET\b"),
}

def indice_categorie(lib):
    citees = [cat for cat, motif in INDICES_CATEGORIE.items() if re.search(motif, lib)]
    return citees[0] if len(citees) == 1 else None

indices = {lib: indice_categorie(lib) for lib in classement}
df["categorie_libelle"] = df["diplome_norm"].map(lambda l: indices[l] if pd.notna(l) else None)
df.loc[df["niveau_theorique"].eq("AUTRES"), "categorie_libelle"] = CAT_AUTRES

df["categorie_libelle"].fillna("(aucun indice : l'école décidera)").value_counts().to_frame("nb_lignes")

,nb_lignes
categorie_libelle,
AUTRES,13771
DIPLOME UNIVERSITAIRE,11298
(aucun indice : l'école décidera),1714
DIPLOME D'INGENIEUR,747
DIPLOME DE COMMERCE,23


---
## Étape 4 : Niveau fiabilisé

Une fonction, `decider_niveau`, croise le niveau saisi (étape 2) et le niveau théorique (étape 3) et dit ce qu'elle a fait :

| Situation | Niveau retenu | Décision |
|---|---|---|
| Libellé de niveau AUTRES (certificat, MBA, executive, métiers de banque) | AUTRES, la saisie est ignorée | `AUTRES` |
| Saisie vide, libellé classé | le niveau théorique | `COMPLETE` |
| Saisie égale au niveau théorique | — | `EGAL` |
| Saisie différente mais compatible | le niveau théorique | `INTERPRETE` (ou `CONVENTION`, sans flag, si `signaler = False`) |
| Saisie incompatible | **le libellé l'emporte** (il vient du diplôme ; le niveau est ressaisi à la main) | `INCOHERENCE` |
| Libellé à fourchette | la saisie si elle tient dans la fourchette, ramenée dedans si elle déborde, sinon le plancher ; sans saisie, le plancher ou `DEFAUT_FOURCHETTE` | idem |
| Pas de niveau théorique | la saisie (un « et plus » est ensuite arrondi au niveau sûr : `BAC+5 ET PLUS` → `BAC+5`, flag `NIVEAU_ARRONDI`) | `SAISIE` |
| Rien des deux | vide | `MANQUANT` |

Chaque ligne reçoit ensuite **`niveau_fiabilite`** et **`motif_niveau`** : `SUR` quand le niveau ne doit rien à un choix (libellé et saisie d'accord, ou l'un des deux absent et l'autre précis, ou une convention) ; `INCERTAIN` quand la règle a tranché à la place de quelqu'un — incohérence, fourchette sans saisie ou saisie hors fourchette, saisie imprécise seule, arrondi d'un « et plus » ; `MANQUANT` sans niveau. Le motif dit ce qui a été retenu et pourquoi. **Seules les lignes SUR comptent dans la vue salarié** (étape 8) ; les autres y apparaissent comme points à discuter.

In [8]:
ARRONDI_NIVEAU = {"BAC+2 ET PLUS": "BAC+2", "BAC+3 OU BAC+4": "BAC+3", "BAC+4 ET PLUS": "BAC+4",
                  "BAC+5 ET PLUS": "BAC+5", "BAC+6 ET PLUS": "BAC+6"}

def decider_niveau(niveau_norm, theo, type_dipl):
    # -> (niveau, décision). niveau_norm = écriture nettoyée du niveau saisi ; theo = niveau théorique du libellé
    ecrit = NIVEAUX_ECRITS.get(niveau_norm) if pd.notna(niveau_norm) else None   # None : rien de saisi (ou écriture inconnue)
    theo = None if pd.isna(theo) else theo
    if theo == "AUTRES":
        return "AUTRES", "AUTRES"
    if type_dipl in FOURCHETTES_NIVEAU:                           # libellé à fourchette : la saisie tranche, sinon le plancher
        if ecrit is None:
            return DEFAUT_FOURCHETTE.get(type_dipl, FOURCHETTES_NIVEAU[type_dipl][0]), "COMPLETE"
        saisi, r_min, r_max = ecrit[0], NIVEAUX[ecrit[1]], NIVEAUX[ecrit[2]]
        f_min, f_max = (NIVEAUX[n] for n in FOURCHETTES_NIVEAU[type_dipl])
        if r_min >= f_min and r_max <= f_max:                 # la saisie tient entièrement dans la fourchette
            return saisi, "EGAL"
        bas, haut = max(r_min, f_min), min(r_max, f_max)
        if bas > haut:                                        # aucun niveau commun : le plancher
            return FOURCHETTES_NIVEAU[type_dipl][0], "INCOHERENCE"
        choix = RANG_VERS_NIVEAU[min(max(NIVEAUX[saisi], bas), haut)]
        return choix, ("EGAL" if choix == saisi else "INTERPRETE")
    if ecrit is None and theo is None:
        return None, "MANQUANT"
    if ecrit is None:
        return theo, "COMPLETE"
    saisi, r_min, r_max, signaler = ecrit[0], NIVEAUX[ecrit[1]], NIVEAUX[ecrit[2]], ecrit[4]
    if theo is None:
        return saisi, "SAISIE"
    if r_min <= NIVEAUX[theo] <= r_max:
        if saisi == theo:
            return theo, "EGAL"
        return theo, ("INTERPRETE" if signaler else "CONVENTION")
    return theo, "INCOHERENCE"

decisions = [decider_niveau(n, t, f) for n, t, f in zip(df["niveau_norm"], df["niveau_theorique"], df["type_diplome"])]
df["niveau_fiabilise"] = [d[0] for d in decisions]
df["decision_niveau"] = [d[1] for d in decisions]
df.loc[df["type_ligne"].eq("CERTIFICATION"), ["niveau_fiabilise", "decision_niveau"]] = ["AUTRES", "AUTRES"]

avant_arrondi = df["niveau_fiabilise"].copy()
df["ctrl_niveau_arrondi"] = avant_arrondi.isin(ARRONDI_NIVEAU)
df["niveau_fiabilise"] = avant_arrondi.replace(ARRONDI_NIVEAU)
df["ctrl_niveau_complete"] = df["decision_niveau"].eq("COMPLETE")
df["ctrl_niveau_interprete"] = df["decision_niveau"].eq("INTERPRETE")
df["ctrl_incoherence_niveau"] = df["decision_niveau"].eq("INCOHERENCE")
df["ctrl_niveau_manquant"] = df["decision_niveau"].eq("MANQUANT")
df["rang_niveau"] = df["niveau_fiabilise"].map(NIVEAUX)

# Le niveau de la ligne est-il SUR, ou le résultat d'un choix (INCERTAIN) ? Et pourquoi. C'est ce qui décide, à l'étape 8,
# de ce qui entre dans la vue salarié : seules les lignes SUR comptent, le reste part en discussion avec son motif.
def qualifier_niveau(saisi, theo, type_dipl, decision, niveau, avant, imprecis):
    fourchette = FOURCHETTES_NIVEAU.get(type_dipl)
    f = f"{fourchette[0]} à {fourchette[1]}" if fourchette else ""
    if decision == "MANQUANT":
        return "MANQUANT", "aucun niveau : ni saisi, ni déductible du libellé"
    if decision == "INCOHERENCE":
        return "INCERTAIN", (f"saisie {saisi} hors fourchette {f} : plancher {niveau} retenu" if fourchette
                             else f"saisie {saisi} incompatible avec le libellé ({theo}) : libellé retenu")
    if fourchette and decision == "COMPLETE":
        return "INCERTAIN", f"libellé à fourchette {f} sans saisie : {niveau} retenu"
    if fourchette and decision == "INTERPRETE":
        return "INCERTAIN", f"saisie imprécise {saisi} ramenée dans la fourchette {f} : {niveau} retenu"
    if decision == "SAISIE" and imprecis:
        return "INCERTAIN", f"niveau saisi imprécis « {saisi} », aucun libellé pour trancher : {niveau} retenu"
    if avant != niveau:
        return "INCERTAIN", f"« {avant} » arrondi au niveau sûr {niveau}"
    return "SUR", ""

qualif = [qualifier_niveau(*t) for t in zip(df["niveau_normalise"], df["niveau_theorique"], df["type_diplome"], df["decision_niveau"],
                                             df["niveau_fiabilise"], avant_arrondi, df["ctrl_niveau_saisi_imprecis"])]
df["niveau_fiabilite"] = [q[0] for q in qualif]
df["motif_niveau"] = [q[1] for q in qualif]

display(pd.crosstab(df["decision_niveau"], df["niveau_fiabilite"], margins=True, margins_name="Total"))
# Les incohérences les plus fréquentes (le niveau retenu est celui du libellé)
(df[df["ctrl_incoherence_niveau"]].groupby(["diplome", "niveau_normalise", "niveau_theorique"]).size()
   .sort_values(ascending=False).rename("nb_lignes").reset_index().head(15))

niveau_fiabilite,INCERTAIN,MANQUANT,SUR,Total
decision_niveau,,,,
AUTRES,0,0,2568,2568
COMPLETE,22,0,10618,10640
CONVENTION,0,0,2232,2232
EGAL,4,0,11333,11337
INCOHERENCE,251,0,0,251
INTERPRETE,6,0,224,230
MANQUANT,0,76,0,76
SAISIE,15,0,204,219
Total,298,76,27179,27553


,diplome,niveau_normalise,niveau_theorique,nb_lignes
0,DIPLOME TECHNICIEN SPECIALISE,BAC+3,BAC+2,53
1,MAITRISE,BAC+5,BAC+4,11
2,LICENCE ANTERIEURE A 2006,BAC+3,BAC+4,8
3,Master,BAC,BAC+5,8
4,TECHNICIEN / OU TECHNICIEN SPECIALISE,BAC+3,BAC+2,8
5,BACCALAUREAT,BAC+2,BAC,8
6,DIPLOME CYCLE SUPERIEUR,BAC+4,BAC+5,7
7,Classes Préparatoires / Ecole de Commerce,BAC+3,BAC+2,6
8,DIPLOME CYCLE NORMAL,BAC+3,BAC+4,6
9,NIVEAU BAC+2,SECONDAIRE,BAC+2,6


---
## Étape 5 : École et catégorie

**École** : `ref_ecoles.xlsx` (onglet `ref_ecoles`) donne, pour chaque code du SIRH, le nom réel, le pays, la **nature** (`ECOLE_INGENIEURS`, `ECOLE_COMMERCE`, `UNIVERSITE`, `AUTRE` = lycée / OFPPT / CFPB / organismes, `NON_PRECISE`) et la **confiance** (`SUR`, `PROBABLE`, `A_CONFIRMER`). Un code absent du référentiel est gardé tel quel, `A_CONFIRMER`, jamais rapproché d'un voisin (`ENSA` ≠ `ENSAM`), et exporté dans `Ecoles_a_completer`. Les écoles génériques (`LYCEE`, `UNIVERSITE MAROCAINE (NON PRECISEE)`) sont signalées, pas bloquées.

**Catégorie du diplôme** (`categorie_diplome`), la première règle qui s'applique décide :

| Ordre | Règle | Exemple |
|---|---|---|
| 1 | l'indice du libellé est ingénieur, commerce ou autres | un « Diplôme d'ingénieur » passé à l'université reste un diplôme d'ingénieur ; un BTS reste dans AUTRES |
| 2 | sinon la nature de l'école (ingénieurs, commerce, université) | un « Master » de l'ISCAE est un diplôme de commerce ; un « Master » de l'EMSI, un diplôme d'ingénieur |
| 3 | sinon `CATEGORIE_PAR_DEFAUT` du type de diplôme | un « Cycle normal » sans école connue est un diplôme de commerce (ISCAE / ENCG) |
| 4 | sinon l'indice universitaire du libellé | une « Licence » sans école connue |
| 5 | sinon AUTRES (signalé `CATEGORIE_NON_TROUVEE` si ni le libellé ni l'école n'ont parlé) | un « Diplôme » sans école |

La règle 2 est un choix : une licence ou un master se préparent aussi en école d'ingénieurs ou de commerce, et c'est l'école qui dit de quel genre de diplôme il s'agit.

**La catégorie est-elle sûre ?** (`categorie_fiabilite`) Oui pour les règles 1 à 3, et la confiance de l'école n'y change rien : un code `A_CONFIRMER` dit qu'on n'est pas sûr de *quel* ISG il s'agit, pas que ce n'est pas une école de gestion — la nature suffit. Oui aussi pour la règle 4 quand le libellé nomme un diplôme universitaire par nature (`UNIV_EXPLICITE` : DUT, DEUG, licence fondamentale, maîtrise, DESS / DESA / DEA, doctorat, « universitaire »). `INCERTAIN` quand elle vient de la règle 4 avec un libellé qui existe aussi hors université (« Master », « Licence », « Licence professionnelle », « Bachelor ») sans école de nature connue, ou de la règle 5 : on a classé par défaut, sans le savoir. `CATEGORIE_INCERTAINE_SANS_ECOLE = False` désactive ce doute (les « Master » sans école sont alors tenus pour universitaires, comme en v12).

In [9]:
ING, COM, UNI, AUT, NP = "ECOLE_INGENIEURS", "ECOLE_COMMERCE", "UNIVERSITE", "AUTRE", "NON_PRECISE"
NATURES = {ING, COM, UNI, AUT, NP}

ref_xl = pd.read_excel(trouver_fichier(FICHIER_REF_ECOLES), sheet_name="ref_ecoles")
ref_xl["code_source"] = ref_xl["code_source"].map(normaliser_texte)     # même nettoyage que les données
ref_xl = ref_xl.dropna(subset=["code_source"]).drop_duplicates("code_source", keep="last")
assert ref_xl["nature"].isin(NATURES).all(), f"nature inconnue dans {FICHIER_REF_ECOLES} : {set(ref_xl['nature']) - NATURES}"
REF_ECOLES = {r.code_source: (None if pd.isna(r.ecole_nom) else r.ecole_nom, None if pd.isna(r.pays) else r.pays, r.nature, r.confiance)
              for r in ref_xl.itertuples()}

def ref_ecole(code):
    if pd.isna(code):
        return (None, None, NP, "SUR")
    return REF_ECOLES.get(code, (code, None, NP, "A_CONFIRMER"))    # hors référentiel : gardé tel quel, à confirmer

res = df["ecole_norm"].map(ref_ecole)
for i, col in enumerate(["ecole_nom", "ecole_pays", "ecole_nature", "ecole_confiance"]):
    df[col] = res.map(lambda t: t[i])
df["ctrl_ecole_manquante"] = df["ecole_nom"].isna()
df["ctrl_ecole_generique"] = df["ecole_nom"].fillna("").str.contains("NON PRECIS")
df["ctrl_ecole_a_confirmer"] = df["ecole_confiance"].eq("A_CONFIRMER") & df["ecole_nom"].notna()
# Ligne de diplôme sans niveau dans une université ou une école post-bac : le diplôme vaut au moins le bac, on ne devine pas plus
df["ctrl_post_bac_sans_niveau"] = df["ctrl_niveau_manquant"] & df["ecole_nature"].isin([ING, COM, UNI])

ecoles_a_completer = (df[df["ecole_norm"].notna() & ~df["ecole_norm"].isin(REF_ECOLES)]
                        .groupby("ecole_norm").agg(nb_lignes=("ligne_source", "size"), exemple_diplome=("diplome", "first"))
                        .reset_index().rename(columns={"ecole_norm": "code_source"}).sort_values("nb_lignes", ascending=False))
print(f"{len(REF_ECOLES)} codes dans le référentiel ; hors référentiel : {len(ecoles_a_completer)} "
      f"(onglet Ecoles_a_completer du fichier de sortie)")
display(ecoles_a_completer.head(15))
pd.crosstab(df["ecole_nature"], df["ecole_confiance"], margins=True, margins_name="Total")

326 codes dans le référentiel ; hors référentiel : 0 (onglet Ecoles_a_completer du fichier de sortie)


,code_source,nb_lignes,exemple_diplome


ecole_confiance,A_CONFIRMER,PROBABLE,SUR,Total
ecole_nature,,,,
AUTRE,28,50,11684,11762
ECOLE_COMMERCE,258,802,3064,4124
ECOLE_INGENIEURS,94,41,757,892
NON_PRECISE,17,1,2949,2967
UNIVERSITE,4,180,7624,7808
Total,401,1074,26078,27553


In [10]:
NATURE_VERS_CATEGORIE = {ING: CAT_ING, COM: CAT_COM, UNI: CAT_UNIV}
CATEGORIE_PAR_DEFAUT = {"DIPLOME CYCLE NORMAL": CAT_COM, "DIPLOME CYCLE SUPERIEUR": CAT_COM}   # diplômes ISCAE / ENCG

indice = df["categorie_libelle"]
par_ecole = df["ecole_nature"].map(NATURE_VERS_CATEGORIE)
par_defaut = df["type_diplome"].map(CATEGORIE_PAR_DEFAUT)
regles = [indice.isin([CAT_ING, CAT_COM, CAT_AUTRES]), par_ecole.notna(), par_defaut.notna(), indice.eq(CAT_UNIV)]
df["categorie_diplome"] = np.select(regles, [indice, par_ecole, par_defaut, CAT_UNIV], default=CAT_AUTRES)
regle_cat = np.select(regles, [1, 2, 3, 4], default=5)
df["ctrl_categorie_non_trouvee"] = (regle_cat == 5) & ~df["ecole_nature"].eq(AUT)

# Catégorie sûre ou choisie par défaut ? Un « Master » sans école de nature connue peut venir d'une université
# comme d'une école de commerce ; un DUT ou une maîtrise sont universitaires par nature.
CATEGORIE_INCERTAINE_SANS_ECOLE = True
UNIV_EXPLICITE = (r"UNIVERSIT|D\.?E\.?U\.?G|\bD\.?U\.?T\b|\bDEUST\b|LICENCE FONDAMENTALE|MAITRISE|DOCTORAT|\bPH\.?\s?D\b|DOCTEUR"
                  r"|\bD\.?E\.?S\.?S\b|\bD\.?E\.?A\b|\bD\.?E\.?S\.?A\b|ETUDES SUPERIEURE?S|ETUDES APPROFONDIES|(?:3\s*E(?:ME)?|TROISIEME) CYCLE")
univ_explicite = df["diplome_norm"].fillna("").str.contains(UNIV_EXPLICITE, regex=True)
incertaine = ((regle_cat == 4) & ~univ_explicite & CATEGORIE_INCERTAINE_SANS_ECOLE) | df["ctrl_categorie_non_trouvee"]
df["categorie_fiabilite"] = np.where(df["type_ligne"].eq("DIPLOME") & incertaine, "INCERTAIN", "SUR")
df["motif_categorie"] = np.where(df["categorie_fiabilite"].eq("INCERTAIN"), "catégorie par défaut, école de nature inconnue (" + df["ecole_nom"].fillna("école absente") + ")", "")

display(pd.crosstab(df["categorie_diplome"], df["categorie_fiabilite"], margins=True, margins_name="Total"))

(pd.crosstab(df["niveau_fiabilise"].fillna("(vide)"), df["categorie_diplome"], margins=True, margins_name="Total")
   .reindex(list(dict.fromkeys(NIVEAUX))[::-1] + ["AUTRES", "(vide)", "Total"]).dropna(how="all").astype(int))

categorie_fiabilite,INCERTAIN,SUR,Total
categorie_diplome,,,
AUTRES,135,13988,14123
DIPLOME D'INGENIEUR,0,1003,1003
DIPLOME DE COMMERCE,0,3827,3827
DIPLOME UNIVERSITAIRE,795,7805,8600
Total,930,26623,27553


categorie_diplome,AUTRES,DIPLOME D'INGENIEUR,DIPLOME DE COMMERCE,DIPLOME UNIVERSITAIRE,Total
niveau_fiabilise,,,,,
EXPERTISE COMPTABLE,2,0,4,1,7
BAC+7,0,2,3,20,25
BAC+6,0,0,3,4,7
BAC+5,12,905,2526,2249,5692
BAC+4,249,45,925,762,1981
BAC+3,82,44,320,4218,4664
BAC+2,2714,4,26,1306,4050
BAC+1,0,0,0,1,1
BAC,8446,0,0,2,8448


---
## Étape 6 : Dédoublonnage

Après normalisation (avant, `Bac + 5` et `BAC+5` empêchent de reconnaître deux lignes identiques). Trois niveaux :

1. **Doublons** : même matricule, libellé nettoyé, niveau fiabilisé, spécialité nettoyée et école. Les lignes sont d'abord triées de la mieux renseignée à la moins renseignée (niveau saisi précis > imprécis > vide), puis la première est gardée : on ne perd plus un niveau précis au profit d'un « NIVEAU UNIVERSITAIRE ».
2. **Doublons partiels** : même matricule, même niveau, même libellé — ou même **type de diplôme** quand ses libellés sont de simples synonymes (`TYPES_SYNONYMES` : `ITB` = `INSTITUT DES TECHNIQUES DE BANQUE`, `BAC` = `BACCALAUREAT`…) — et la ligne n'apporte rien de plus qu'une autre (spécialité et école vides ou identiques). Retirée aussi : elle ne contient aucune information propre. Ex. `Brevet Bancaire` sans école à côté de `BREVET BANCAIRE / C.F.P.B.`. Un « Master Compliance » n'est jamais retiré au profit d'un « MASTER » : le libellé porte une information.
3. **Quasi-doublons** (`QUASI_DOUBLON_A_ARBITRER`, conservés) : même matricule, même type de diplôme ou même libellé, même niveau, mais spécialité ou école différentes. Sans le dossier on ne sait pas laquelle est la bonne.

Toutes les lignes retirées vont dans l'onglet `Doublons_supprimes` avec leur motif et leur ligne d'origine.

In [11]:
CLES_DOUBLON = ["matricule", "diplome_norm", "niveau_fiabilise", "specialite_norm", "ecole_nom"]
CLES_TYPE = ["matricule", "type_diplome", "niveau_fiabilise"]
# Types de diplôme dont les libellés sont de simples synonymes (ITB = INSTITUT DES TECHNIQUES DE BANQUE, BAC = BACCALAUREAT…) :
# deux libellés différents y désignent le même diplôme. Ailleurs (MASTER, LICENCE…) le libellé peut porter la spécialité
# (« Master Compliance » ≠ « MASTER ») : un doublon partiel exige alors le même libellé.
TYPES_SYNONYMES = {"ITB", "BACCALAUREAT", "INGENIEUR", "MBA", "BTS / TECHNICIEN SPECIALISE", "LICENCE (ANCIEN REGIME 4 ANS)",
                      "MAITRISE", "DIPLOME CYCLE NORMAL", "DIPLOME CYCLE SUPERIEUR", "SECONDAIRE / QUALIFIANT", "DOCTORAT",
                      "DIPLOME DES METIERS DE BANQUE (SANS EQUIVALENCE)"}
nb_avant = len(df)

# 1. Doublons : la ligne la mieux renseignée est gardée
qualite = df["niveau_norm"].isna().astype(int) * 2 + df["ctrl_niveau_saisi_imprecis"].astype(int)   # 0 = saisie précise
df = df.assign(_q=qualite).sort_values(["_q", "ligne_source"])
dup = df.duplicated(subset=CLES_DOUBLON, keep="first")
doublons = df[dup].assign(motif="doublon")
df = df[~dup].drop(columns="_q").sort_values("ligne_source")

# 2. Doublons partiels : même type de diplôme et niveau, sans information propre face à une autre ligne du salarié
def _partiels(g):
    typ = g.name[1]                                            # le type de diplôme, constant dans le groupe
    rows = g[["specialite_norm", "ecole_nom", "diplome_norm"]].to_numpy(dtype=object)
    out = []
    for i, (s, e, lib) in enumerate(rows):
        domine = False
        for j, (s2, e2, lib2) in enumerate(rows):
            if i == j or not (lib == lib2 or typ in TYPES_SYNONYMES):
                continue
            compat = (pd.isna(s) or s == s2) and (pd.isna(e) or e == e2)
            plus = (pd.isna(s) and pd.notna(s2)) or (pd.isna(e) and pd.notna(e2))
            if compat and plus:
                domine = True
                break
        out.append(domine)
    return pd.Series(out, index=g.index)

multi = df[df["niveau_fiabilise"].notna()].groupby(CLES_TYPE).filter(lambda g: len(g) > 1)
partiel = (multi.groupby(CLES_TYPE, group_keys=False)[["specialite_norm", "ecole_nom", "diplome_norm"]].apply(_partiels)
           if len(multi) else pd.Series(dtype=bool))
partiel = partiel.reindex(df.index, fill_value=False).astype(bool)
doublons = pd.concat([doublons, df[partiel].assign(motif="doublon partiel (ligne moins renseignée)")])
df = df[~partiel].copy()

# 3. Quasi-doublons : même type de diplôme (ITB = INSTITUT DES TECHNIQUES DE BANQUE, BAC = BACCALAUREAT…) ou même libellé, même niveau
est_diplome = df["type_ligne"].eq("DIPLOME") & df["niveau_fiabilise"].notna()
df["ctrl_quasi_doublon"] = est_diplome & (df.duplicated(subset=CLES_TYPE, keep=False)
                                          | df.duplicated(subset=["matricule", "diplome_norm", "niveau_fiabilise"], keep=False))

pd.Series({"Lignes avant": nb_avant,
           "Doublons retirés": int(doublons["motif"].eq("doublon").sum()),
           "Doublons partiels retirés": int(doublons["motif"].ne("doublon").sum()),
           "Lignes après": len(df),
           "Quasi-doublons conservés (à arbitrer)": int(df["ctrl_quasi_doublon"].sum()),
           "Salariés concernés par un quasi-doublon": df.loc[df["ctrl_quasi_doublon"], "matricule"].nunique()}).to_frame("valeur")

,valeur
Lignes avant,27553
Doublons retirés,6423
Doublons partiels retirés,339
Lignes après,20791
Quasi-doublons conservés (à arbitrer),496
Salariés concernés par un quasi-doublon,236


---
## Étape 7 : Contrôles et statut

**Bloquant** : une décision humaine est nécessaire. **Informatif** : une transformation a été faite, la ligne reste exploitable. Le statut retient le motif bloquant le plus important (ordre de `PRIORITE_STATUT`), sinon `OK`. Un niveau `INCERTAIN` (étape 4) est bloquant : `A VERIFIER - niveau incertain`. Une catégorie `INCERTAIN` (étape 5) l'est aussi : `A COMPLETER - école (catégorie)`, car c'est l'école qui manque. `ECOLE_A_CONFIRMER` n'est plus bloquant : un code d'école obscur ne change ni le niveau ni, si sa nature est connue, la catégorie.

In [12]:
CONTROLES = {   # colonne : (code, nature)
    "ctrl_diplome_manquant":         ("DIPLOME_MANQUANT", "bloquant"),
    "ctrl_niveau_manquant":          ("NIVEAU_MANQUANT", "bloquant"),
    "ctrl_incoherence_niveau":       ("INCOHERENCE_NIVEAU", "bloquant"),
    "ctrl_quasi_doublon":            ("QUASI_DOUBLON_A_ARBITRER", "bloquant"),
    "ctrl_ecole_a_confirmer":        ("ECOLE_A_CONFIRMER", "informatif"),
    "ctrl_post_bac_sans_niveau":     ("ECOLE_POST_BAC_SANS_NIVEAU", "informatif"),
    "ctrl_libelle_non_diplomant":    ("LIBELLE_NON_DIPLOMANT", "informatif"),
    "ctrl_diplome_non_classe":       ("DIPLOME_NON_CLASSE", "informatif"),
    "ctrl_libelle_ambigu":           ("LIBELLE_A_FOURCHETTE", "informatif"),
    "ctrl_niveau_sans_equivalence":  ("NIVEAU_SANS_EQUIVALENCE", "informatif"),
    "ctrl_niveau_saisi_imprecis":    ("NIVEAU_SAISI_IMPRECIS", "informatif"),
    "ctrl_niveau_complete":          ("NIVEAU_COMPLETE_PAR_LIBELLE", "informatif"),
    "ctrl_niveau_interprete":        ("NIVEAU_SAISI_INTERPRETE", "informatif"),
    "ctrl_niveau_arrondi":           ("NIVEAU_ARRONDI", "informatif"),
    "ctrl_ecole_manquante":          ("ECOLE_MANQUANTE", "informatif"),
    "ctrl_ecole_generique":          ("ECOLE_GENERIQUE", "informatif"),
    "ctrl_categorie_non_trouvee":    ("CATEGORIE_NON_TROUVEE", "informatif"),
}
for c in CONTROLES:
    df[c] = df[c].fillna(False).astype(bool)
df["flags"] = [" | ".join(CONTROLES[c][0] for c in CONTROLES if r[c]) for r in df[list(CONTROLES)].to_dict("records")]

PRIORITE_STATUT = [
    (df["ctrl_diplome_manquant"],   "A COMPLETER - diplôme"),
    (df["ctrl_niveau_manquant"],    "A COMPLETER - niveau"),
    (df["ctrl_incoherence_niveau"], "A VERIFIER - niveau saisi ≠ libellé"),
    (df["niveau_fiabilite"].eq("INCERTAIN"), "A VERIFIER - niveau incertain"),
    (df["ctrl_quasi_doublon"],      "A ARBITRER - quasi-doublon"),
    (df["categorie_fiabilite"].eq("INCERTAIN"), "A COMPLETER - école (catégorie)"),
]
df["statut"] = np.select([c for c, _ in PRIORITE_STATUT], [s for _, s in PRIORITE_STATUT], default="OK")

tab_controles = pd.DataFrame([(code, nature, int(df[c].sum())) for c, (code, nature) in CONTROLES.items()],
                             columns=["controle", "nature", "nb_lignes"])
display(tab_controles)
df["statut"].value_counts().to_frame("nb_lignes").assign(**{"% lignes": lambda d: (d["nb_lignes"] / len(df) * 100).round(1)})

,controle,nature,nb_lignes
0,DIPLOME_MANQUANT,bloquant,1
1,NIVEAU_MANQUANT,bloquant,60
2,INCOHERENCE_NIVEAU,bloquant,186
3,QUASI_DOUBLON_A_ARBITRER,bloquant,496
4,ECOLE_A_CONFIRMER,informatif,312
5,ECOLE_POST_BAC_SANS_NIVEAU,informatif,47
6,LIBELLE_NON_DIPLOMANT,informatif,292
7,DIPLOME_NON_CLASSE,informatif,223
8,LIBELLE_A_FOURCHETTE,informatif,259
9,NIVEAU_SANS_EQUIVALENCE,informatif,25


,nb_lignes,% lignes
statut,,
OK,19370,93.2
A COMPLETER - école (catégorie),654,3.1
A ARBITRER - quasi-doublon,480,2.3
A VERIFIER - niveau saisi ≠ libellé,186,0.9
A COMPLETER - niveau,60,0.3
A VERIFIER - niveau incertain,40,0.2
A COMPLETER - diplôme,1,0.0


---
## Étape 8 : Vue salarié

Deux tables. **`Vue_salarie`** ne contient que les salariés dont tout est sûr : on peut s'en servir sans relire. **`Salaries_a_discuter`** reçoit tous les autres, avec un marqueur neutre et la raison, colonne par colonne. Rien, dans aucune des deux, n'est le fruit d'un arbitrage caché.

**Niveau max** : le rang le plus haut parmi les lignes de diplôme dont le niveau est `SUR` (étape 4). Il devient `A VERIFIER` (`NIVEAU_SI_INCERTAIN`) dès qu'une ligne pourrait le changer : une ligne `INCERTAIN` de rang supérieur ou égal au max sûr, ou une ligne sans niveau (`MANQUANT`) — ex. deux « DIPLOME » à l'université et un bac : il a au moins le bac, sûrement plus, on ne devine pas. Une ligne incertaine *sous* le max sûr ne change rien (un master sûr et un BTS incertain : BAC+5). `NON DETERMINE` : aucune ligne de diplôme avec un niveau (certificats, MBA, executive, métiers de banque seulement). `motif_niveau_max` liste les lignes en cause, avec leur motif.

**Catégorie max** : celle des lignes sûres au niveau max. `AUTRES` (`CATEGORIE_SI_AMBIGU`) avec `motif_categorie_max` si ces lignes n'ont pas toutes la même catégorie, si l'une d'elles a une catégorie `INCERTAIN` (étape 5 : classée par défaut faute d'école de nature connue), ou si le niveau max est lui-même à vérifier.

**Diplôme et école affichés** : la ligne sûre au niveau max la plus propre (statut OK, école la plus sûre, puis l'ordre du fichier) ; vides quand le niveau max est à vérifier (les candidats sont dans le motif).

Un salarié passe en discussion dès qu'il a un motif, sur le niveau ou sur la catégorie. `NON DETERMINE` reste dans la vue : ce n'est pas un doute, c'est un fait (aucun diplôme sur l'échelle, par décision — certificats, MBA, executive). Les certifications ne comptent pas (niveau `AUTRES`, sans rang) mais sont comptées à part.

In [13]:
NIVEAU_SI_INCERTAIN = "A VERIFIER"       # le niveau max dépend d'une ligne incertaine ou sans niveau
CATEGORIE_SI_AMBIGU = CAT_AUTRES         # plusieurs catégories au niveau max, ou catégorie par défaut faute d'école
PRIORITE_CONFIANCE = {"SUR": 0, "PROBABLE": 1, "A_CONFIRMER": 2}
index_mat = sorted(df["matricule"].dropna().unique())

dipl = df[df["type_ligne"].eq("DIPLOME")].copy()
dipl["detail"] = dipl["diplome"] + " (" + dipl["niveau_fiabilise"].fillna("sans niveau") + ") : " + dipl["motif_niveau"]
dipl.loc[dipl["ctrl_post_bac_sans_niveau"], "detail"] += " ; école post-bac, donc au moins le bac"
surs = dipl[dipl["niveau_fiabilite"].eq("SUR") & dipl["rang_niveau"].notna()]
incertains = dipl[dipl["niveau_fiabilite"].eq("INCERTAIN") & dipl["rang_niveau"].notna()]
manquants = dipl[dipl["niveau_fiabilite"].eq("MANQUANT")]

# 1. Niveau max sûr, et ce qui pourrait le remettre en cause
rang_sur = surs.groupby("matricule")["rang_niveau"].max().reindex(index_mat)
en_cause = pd.concat([incertains.merge(rang_sur.rename("rang_sur"), left_on="matricule", right_index=True, how="left")
                                .pipe(lambda d: d[d["rang_sur"].isna() | d["rang_niveau"].ge(d["rang_sur"])]),
                      manquants])
motif_niveau_max = en_cause.groupby("matricule")["detail"].agg(" | ".join).reindex(index_mat).fillna("")
a_verifier = motif_niveau_max.ne("")

# 2. Ligne affichée et catégorie : parmi les lignes sûres au niveau max
au_max = surs[surs["rang_niveau"].eq(surs["matricule"].map(rang_sur))].copy()
au_max["_ok"] = au_max["statut"].ne("OK"); au_max["_conf"] = au_max["ecole_confiance"].map(PRIORITE_CONFIANCE)
au_max = au_max.sort_values(["matricule", "_ok", "_conf", "ligne_source"])
top = au_max.drop_duplicates("matricule").set_index("matricule")
au_max["cat_detail"] = au_max["diplome"] + " → " + au_max["categorie_diplome"]
au_max["cat_incertaine"] = au_max["categorie_fiabilite"].eq("INCERTAIN")
au_max["cat_motif"] = np.where(au_max["cat_incertaine"], au_max["diplome"] + " : " + au_max["motif_categorie"], "")
par_mat = au_max.groupby("matricule").agg(n_categories=("categorie_diplome", "nunique"), cat_detail=("cat_detail", lambda s: " ; ".join(dict.fromkeys(s))),
                                         incertaine=("cat_incertaine", "any"),
                                         motif_incertaine=("cat_motif", lambda s: " ; ".join(dict.fromkeys(x for x in s if x)))).reindex(index_mat)
motif_cat = np.select(
    [a_verifier, par_mat["n_categories"].gt(1), par_mat["incertaine"].eq(True)],
    ["niveau max à vérifier", "catégories différentes au niveau max : " + par_mat["cat_detail"].fillna(""),
     par_mat["motif_incertaine"].fillna("")], default="")

# 3. La vue
vue = pd.DataFrame({
    "niveau_max": top["niveau_fiabilise"], "categorie_diplome_max": top["categorie_diplome"],
    "diplome_max": top["diplome"], "ecole_max": top["ecole_nom"],
    "nb_diplomes": dipl.groupby("matricule").size(),
    "nb_certifications": df[df["type_ligne"].eq("CERTIFICATION")].groupby("matricule").size(),
    "nb_lignes_a_traiter": df[df["statut"].ne("OK")].groupby("matricule").size(),
    "nb_lignes_au_max": au_max.groupby("matricule").size(),        # lignes sûres au niveau max
}).reindex(index_mat)
vue["motif_niveau_max"] = motif_niveau_max
vue["motif_categorie_max"] = motif_cat
vue.loc[a_verifier, ["niveau_max", "diplome_max", "ecole_max"]] = [NIVEAU_SI_INCERTAIN, None, None]
vue["niveau_max"] = vue["niveau_max"].fillna("NON DETERMINE")
vue.loc[vue["motif_categorie_max"].ne("") | vue["categorie_diplome_max"].isna(), "categorie_diplome_max"] = CATEGORIE_SI_AMBIGU
vue.loc[vue["niveau_max"].eq("NON DETERMINE"), "categorie_diplome_max"] = None
for c in ["nb_diplomes", "nb_certifications", "nb_lignes_a_traiter", "nb_lignes_au_max"]:
    vue[c] = vue[c].fillna(0).astype(int)
vue = vue.rename_axis("matricule").reset_index()[[
    "matricule", "niveau_max", "motif_niveau_max", "categorie_diplome_max", "motif_categorie_max",
    "diplome_max", "ecole_max", "nb_diplomes", "nb_certifications", "nb_lignes_a_traiter", "nb_lignes_au_max"]]
# La vue ne garde que les salariés sans aucun motif : tout ce qui s'y trouve est sûr. Les autres partent en discussion.
en_discussion = vue["motif_niveau_max"].ne("") | vue["motif_categorie_max"].ne("")
a_discuter = vue[en_discussion].copy()
vue = vue[~en_discussion].drop(columns=["motif_niveau_max", "motif_categorie_max"])

print(f"{len(vue) + len(a_discuter):,} salariés : {len(vue):,} dans la vue (sûrs), {len(a_discuter):,} à discuter".replace(",", " "))
display(vue["niveau_max"].value_counts().reindex(list(dict.fromkeys(NIVEAUX))[::-1] + ["NON DETERMINE"]).dropna().astype(int).to_frame("nb_salaries_vue"))
pd.DataFrame({"niveau_max": a_discuter["niveau_max"].value_counts(), "motif_categorie": a_discuter["motif_categorie_max"].str.split(" :").str[0].value_counts()}).fillna(0).astype(int)

8 864 salariés : 7 928 dans la vue (sûrs)  936 à discuter


,nb_salaries_vue
niveau_max,
EXPERTISE COMPTABLE,3
BAC+7,17
BAC+6,3
BAC+5,3471
BAC+4,753
BAC+3,1969
BAC+2,1292
BAC,164
SECONDAIRE,24


,niveau_max,motif_categorie
A VERIFIER,184,0
BAC + 2,0,7
BAC + 3,0,7
BAC+2,34,0
BAC+3,192,8
BAC+4,44,0
BAC+5,478,0
BAC+6,2,0
BACHELOR BAC+3,0,6
BACHELOR BAC+4,0,3


In [14]:
# Contrôle : le cas « DIPLOME, DIPLOME, NIVEAU BAC » — une ligne sans niveau dans une école post-bac rend le niveau max à vérifier
cas = a_discuter[a_discuter["motif_niveau_max"].str.contains("école post-bac")]
print(f"{len(cas)} salariés")
m = cas["matricule"].iloc[0] if len(cas) else None
display(cas.head(3))
df.loc[df["matricule"].eq(m), ["matricule", "diplome", "niveau", "niveau_fiabilise", "niveau_fiabilite", "motif_niveau", "ecole_nom", "ecole_nature", "statut"]]

46 salariés


,matricule,niveau_max,motif_niveau_max,categorie_diplome_max,motif_categorie_max,diplome_max,ecole_max,nb_diplomes,nb_certifications,nb_lignes_a_traiter,nb_lignes_au_max
763,MATRICULE000764,A VERIFIER,"DIPLOME (sans niveau) : aucun niveau : ni saisi, ni déductible du libellé ; école post...",AUTRES,niveau max à vérifier,None,None,2,0,1,1
1141,MATRICULE001142,A VERIFIER,"DIPLOME (sans niveau) : aucun niveau : ni saisi, ni déductible du libellé ; école post...",AUTRES,niveau max à vérifier,None,None,3,1,1,1
1846,MATRICULE001847,A VERIFIER,"DIPLOME (sans niveau) : aucun niveau : ni saisi, ni déductible du libellé ; école post...",AUTRES,niveau max à vérifier,None,None,3,0,1,1


,matricule,diplome,niveau,niveau_fiabilise,niveau_fiabilite,motif_niveau,ecole_nom,ecole_nature,statut
1457,MATRICULE000764,DIPLOME,NaN,None,MANQUANT,"aucun niveau : ni saisi, ni déductible du libellé",INSTITUT DE FORMATION EN INFORMATIQUE ET GESTION (A PRECISER),ECOLE_COMMERCE,A COMPLETER - niveau
1458,MATRICULE000764,LICENCE ANTERIEURE A 2006,BAC+4,BAC+4,SUR,,UNIVERSITE MAROCAINE (NON PRECISEE),UNIVERSITE,OK


---
## Étape 9 : Export

| Onglet | Contenu |
|---|---|
| `Synthese` | indicateurs avant / après |
| `BDD_Diplomes` | la base fiabilisée : diplôme, type de diplôme, catégorie, niveau (écrit, théorique, retenu, fiabilité et motif), école et spécialité (source et dernière version), statut, flags |
| `A_traiter` | les lignes dont le statut n'est pas OK |
| `Vue_salarie` | une ligne par salarié **sûr** : `Niveau_Etude_Max`, `Cat_Ecole_Max`, diplôme, école, compteurs — utilisable sans relecture (`RENOMMAGE_VUE` pour les noms de colonnes) |
| `Salaries_a_discuter` | les autres salariés, avec `A VERIFIER` / `AUTRES` et le motif, colonne par colonne |
| `Doublons_supprimes` | les lignes retirées, avec motif et ligne d'origine |
| `Controles` | nombre de lignes par contrôle |
| `Ecoles_a_completer` | codes école absents de `ref_ecoles.xlsx` (à recopier dans le référentiel) |
| `Libelles_a_classer` | libellés non classés ou non diplômants, avec effectifs |
| `REF_diplomes`, `REF_ecoles` | les correspondances source → fiabilisé, avec effectifs |

In [15]:
bdd = df[["matricule", "type_ligne", "diplome", "type_diplome", "categorie_diplome",
          "niveau", "niveau_theorique", "niveau_fiabilise", "niveau_fiabilite", "motif_niveau", "categorie_fiabilite", "motif_categorie",
          "specialite", "specialite_norm",
          "ecole", "ecole_nom", "ecole_pays", "ecole_nature", "ecole_confiance",
          "statut", "flags", "ligne_source"]].rename(columns={
    "niveau": "niveau_ecrit", "niveau_fiabilise": "niveau", "specialite": "specialite_source", "ecole": "ecole_source"})

ref_diplomes = (bdd.groupby(["diplome", "type_diplome", "categorie_diplome", "niveau"], dropna=False)
                   .size().rename("nb_lignes").reset_index().sort_values("nb_lignes", ascending=False))
ref_ecoles_out = (bdd.groupby(["ecole_source", "ecole_nom", "ecole_pays", "ecole_nature", "ecole_confiance"], dropna=False)
                     .size().rename("nb_lignes").reset_index().sort_values("nb_lignes", ascending=False))
libelles_a_classer = (bdd[df["ctrl_diplome_non_classe"] | df["ctrl_libelle_non_diplomant"]]
                        .groupby(["diplome", "type_diplome", "niveau"], dropna=False).size().rename("nb_lignes")
                        .reset_index().sort_values("nb_lignes", ascending=False))

synthese = pd.DataFrame([
    ("Lignes", len(brut), len(bdd)),
    ("Salariés", brut["Matricule fictif"].nunique(), len(vue) + len(a_discuter)),
    ("Écritures de niveau distinctes", brut["Niveau du diplôme"].nunique(), bdd["niveau"].nunique()),
    ("Lignes diplôme sans niveau", int(brut["Niveau du diplôme"].isna().sum()), int(df["ctrl_niveau_manquant"].sum())),
    ("Écoles distinctes", brut["Ecole Diplome ou Organisme Certification"].nunique(), bdd["ecole_nom"].nunique()),
    ("Lignes OK", None, int(bdd["statut"].eq("OK").sum())),
    ("Lignes à traiter", None, int(bdd["statut"].ne("OK").sum())),
    ("Lignes diplôme au niveau SUR", None, int(df["niveau_fiabilite"].eq("SUR").sum())),
    ("Salariés dans la vue (sûrs)", None, len(vue)),
    ("  dont niveau max NON DETERMINE", None, int(vue["niveau_max"].eq("NON DETERMINE").sum())),
    ("Salariés à discuter", None, len(a_discuter)),
    ("  dont niveau max A VERIFIER", None, int(a_discuter["niveau_max"].eq(NIVEAU_SI_INCERTAIN).sum())),
    ("  dont catégorie max ambiguë", None, int(a_discuter["motif_categorie_max"].ne("").sum())),
], columns=["indicateur", "avant", "apres"])

# Noms de colonnes attendus dans le fichier de sortie (les noms internes du notebook ne changent pas)
RENOMMAGE_VUE = {"niveau_max": "Niveau_Etude_Max", "categorie_diplome_max": "Cat_Ecole_Max"}

onglets = {
    "Synthese": synthese,
    "BDD_Diplomes": bdd,
    "A_traiter": bdd[bdd["statut"].ne("OK")].sort_values(["statut", "matricule"]),
    "Vue_salarie": vue.rename(columns=RENOMMAGE_VUE),
    "Salaries_a_discuter": a_discuter.rename(columns=RENOMMAGE_VUE),
    "Doublons_supprimes": doublons[["ligne_source", "motif"] + COLS].sort_values("ligne_source"),
    "Controles": tab_controles,
    "Ecoles_a_completer": ecoles_a_completer.assign(ecole_nom=None, pays=None, nature=None, confiance=None),
    "Libelles_a_classer": libelles_a_classer,
    "REF_diplomes": ref_diplomes,
    "REF_ecoles": ref_ecoles_out,
}

CHEMIN_SORTIE = DOSSIER / FICHIER_SORTIE
if CHEMIN_SORTIE.exists():
    try:
        open(CHEMIN_SORTIE, "r+b").close()
    except PermissionError:
        raise PermissionError(f"{FICHIER_SORTIE} est sans doute ouvert dans Excel : le fermer, puis relancer cette cellule.") from None

with pd.ExcelWriter(CHEMIN_SORTIE, engine="openpyxl") as xw:
    for nom, d in onglets.items():
        d.to_excel(xw, sheet_name=nom, index=False)
        ws = xw.sheets[nom]
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        for col in ws.columns:
            largeur = max((len(str(c.value)) for c in col[:300] if c.value is not None), default=8)
            ws.column_dimensions[col[0].column_letter].width = min(max(10, largeur + 2), 55)

print(f"Fichier écrit : {CHEMIN_SORTIE.name}")
for nom, d in onglets.items():
    print(f"  {nom:<20} {len(d):>6} lignes")
synthese

Fichier écrit : BDD_Diplomes_FINAL.xlsx
  Synthese                 13 lignes
  BDD_Diplomes          20791 lignes
  A_traiter              1421 lignes
  Vue_salarie            7928 lignes
  Salaries_a_discuter     936 lignes
  Doublons_supprimes     6762 lignes
  Controles                17 lignes
  Ecoles_a_completer        0 lignes
  Libelles_a_classer       73 lignes
  REF_diplomes            354 lignes
  REF_ecoles              325 lignes


,indicateur,avant,apres
0,Lignes,27553.0,20791
1,Salariés,8864.0,8864
2,Écritures de niveau distinctes,25.0,11
3,Lignes diplôme sans niveau,11799.0,60
4,Écoles distinctes,334.0,259
5,Lignes OK,NaN,19370
6,Lignes à traiter,NaN,1421
7,Lignes diplôme au niveau SUR,NaN,20505
8,Salariés dans la vue (sûrs),NaN,7928
9,dont niveau max NON DETERMINE,NaN,232


---
## Pour la prochaine campagne

1. Déposer l'extraction à côté du notebook, changer `FICHIER_SOURCE`, *Run > Run All Cells*.
2. Regarder trois listes : **niveaux écrits inconnus** (étape 2 → `NIVEAUX_ECRITS`), **libellés non classés** (étape 3 et onglet `Libelles_a_classer` → `REF_DIPLOMES_MANUEL` ou `REGLES_DIPLOME`), **écoles hors référentiel** (onglet `Ecoles_a_completer` → `ref_ecoles.xlsx`).
3. Traiter l'onglet `A_traiter` (lignes) et `Salaries_a_discuter` (salariés) : ce sont les seuls cas où un retour au dossier salarié est nécessaire. `Vue_salarie` s'utilise sans relecture.

| Besoin | Où |
|---|---|
| Un vide déguisé de plus | `PLACEHOLDERS` (étape 1) |
| Un nouveau niveau écrit, ou ne plus signaler une convention | `NIVEAUX_ECRITS` (étape 2, colonne `signaler`) |
| Classer un libellé | `REF_DIPLOMES_MANUEL` (cas isolé) ou `REGLES_DIPLOME` (règle) — étape 3 |
| Un libellé à fourchette | `FOURCHETTES_NIVEAU`, `DEFAUT_FOURCHETTE` |
| Décoder, confirmer ou reclasser une école | `ref_ecoles.xlsx` |
| Catégorie par défaut d'un type de diplôme sans école | `CATEGORIE_PAR_DEFAUT` (étape 5) |
| Ce qui rend un niveau de ligne incertain | `qualifier_niveau` (étape 4) |
| Marqueurs neutres de la vue salarié | `NIVEAU_SI_INCERTAIN`, `CATEGORIE_SI_AMBIGU` (étape 8) |